# Практическая работа №3. Задания — Вариант 7

Выполняются все пункты варианта 7 по последнему номеру зачетной книжки:

1. Считать CSV-файлы заказов по частям и сразу фильтровать только строки со статусом `paid`.
2. На основании отфильтрованных данных посчитать: общую выручку, количество оплаченных заказов, средний чек.
3. Объединить результат с таблицей товаров из Excel и посчитать среднюю цену по категориям.
4. Сохранить DataFrame с только «оплаченными» позициями в новый CSV.
5. Оценить, сколько строк было отброшено при фильтрации.

## Подготовка

Импорт библиотек и список файлов с заказами.

In [1]:
import pandas as pd
import numpy as np

orders_files = [
    "../shared/data/orders_2024_01.csv",
    "../shared/data/orders_2024_02.csv",
    "../shared/data/orders_2024_03.csv",
]

### Задание 1. Считать CSV-файлы заказов по частям и сразу фильтровать только строки со статусом `paid`

In [2]:
chunk_size = 1_000
total_rows = 0
paid_chunks = []

for f in orders_files:
    for chunk in pd.read_csv(f, chunksize=chunk_size):
        total_rows += len(chunk)
        paid_chunk = chunk[chunk["status"] == "paid"].copy()
        paid_chunks.append(paid_chunk)

df_paid = pd.concat(paid_chunks, ignore_index=True)
print("Всего строк прочитано:", total_rows)
print("Строк со статусом paid:", df_paid.shape[0])
print(df_paid.head())

Всего строк прочитано: 15000
Строк со статусом paid: 12032
   order_id  user_id       order_datetime  product_id     price  quantity  \
0         2     1079  2024-01-03 17:45:43          16   9172.60         1   
1         3     1041  2024-01-06 22:46:38          25   9035.26         2   
2         4     1083  2024-01-08 21:03:51           9  15234.33         3   
3         5     1068  2024-01-03 23:26:53          24   4134.38         2   
4         6     1069  2024-01-15 13:09:44          28  19359.94         3   

  status  
0   paid  
1   paid  
2   paid  
3   paid  
4   paid  


### Задание 2. Посчитать общую выручку, количество оплаченных заказов и средний чек

In [3]:
df_paid["revenue"] = df_paid["price"] * df_paid["quantity"]

total_revenue = df_paid["revenue"].sum()
paid_orders_count = df_paid["order_id"].nunique()
avg_check = total_revenue / paid_orders_count

print("Общая выручка:", total_revenue)
print("Количество оплаченных заказов:", paid_orders_count)
print("Средний чек:", avg_check)

Общая выручка: 307828994.03
Количество оплаченных заказов: 12032
Средний чек: 25584.191658078456


### Задание 3. Объединить результат с таблицей товаров из Excel и посчитать среднюю цену по категориям

In [4]:
df_products = pd.read_excel("../shared/data/products.xlsx", sheet_name="products")

df_paid_products = df_paid.merge(df_products, on="product_id", how="left")

avg_price_by_category = df_paid_products.groupby("category_name")["price"].mean()
print(avg_price_by_category)

category_name
Electronics    10157.588460
Fashion        10272.667862
Home            9874.648267
Sport          10300.223563
Toys           10141.048013
Name: price, dtype: float64


### Задание 4. Сохранить DataFrame с только «оплаченными» позициями в новый CSV

In [5]:
df_paid.to_csv("paid_orders.csv", index=False)
print("Сохранено строк:", df_paid.shape[0])

Сохранено строк: 12032


### Задание 5. Оценить, сколько строк было отброшено при фильтрации

In [6]:
dropped_rows = total_rows - df_paid.shape[0]
dropped_share = dropped_rows / total_rows * 100

print("Отброшено строк:", dropped_rows)
print(f"Доля отброшенных строк: {dropped_share:.1f}%")

Отброшено строк: 2968
Доля отброшенных строк: 19.8%
